# Clustering of H-Bond Frequencies and RMSD

**What this notebook does:**
This plotting notebook provides two complementary views of simulation clustering:

**Part A – H-bond frequency clustering**
- Reads interaction CSV files for V7 and V12
- Computes per-phase H-bond frequencies in a transition-state window
- Filters to bonds with ≥ 80 % frequency in at least one phase
- Trains a **Decision Tree** and draws a **dendrogram** clustering simulation phases
  by interaction profile
- Performs **PCA** on per-frame interaction data

**Part B – RMSD-based clustering**
- Reads a pre-computed `trans_st_RMSD.csv` (RMSD values for V7 and V12 in the
  transition window)
- Clusters simulation phases by RMSD profile (dendrogram)
- Clusters individual frames by RMSD value (hierarchical + scatter)

**Sections:**
1. Install & import libraries
2. Configuration
3. Part A — H-bond frequency clustering
4. Part B — RMSD-based clustering
5. Part B — PCA on frame-level data


In [ ]:
# 1. Install required libraries
!pip install graphviz
!pip install pydotplus

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Imports
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, export_graphviz, plot_tree
from sklearn.model_selection import train_test_split
from io import StringIO

try:
    import pydotplus
    from IPython.display import Image
    _PYDOT_AVAILABLE = True
except ImportError:
    _PYDOT_AVAILABLE = False

In [ ]:
# 3. Configuration
ROOTDIR_INTERACTIONS = '/content/drive/MyDrive/M1_STAGE/Data/interactions/'
ROOTDIR_TABLES       = '/content/drive/MyDrive/M1_STAGE/Manips/Tables'
OUTPUT_DIR           = '/content/drive/MyDrive/M1_STAGE/Manips/Tables'

INTERACTION_FILES = ['res_V12.csv', 'res_V7.csv']
SIMULATION_NAMES  = ['V12', 'V7']

PHASE_LIMIT_DICO = {
    'V7': 198, 'V8': 180, 'V21': 155,
    'V12': 35, 'V11': 1000, 'V1': 517,
}

SELECTED_FRAMES_DICO = {
    'V12': range(0,  176),
    'V7':  range(75, 251),
}

PHASES_LIST_ORDERED = ['V12_ph1', 'V12_ph2', 'V7_ph1', 'V7_ph2']

def find_hbonds(df):
    return [c for c in df.columns if 'hb' in c]

In [ ]:
# 4. Build H-bond frequency table (transition-state window)
os.chdir(ROOTDIR_INTERACTIONS)
all_phase_frequencies = {}

for i, sim_file in enumerate(INTERACTION_FILES):
    sim = SIMULATION_NAMES[i]
    df_full  = pd.read_csv(sim_file)
    valid    = [ix for ix in df_full.index if ix in SELECTED_FRAMES_DICO.get(sim, [])]
    df_ts    = df_full.loc[valid]
    lim      = PHASE_LIMIT_DICO.get(sim)
    ph1_idx  = df_ts.index[df_ts.index <= lim]
    ph2_idx  = df_ts.index[df_ts.index >  lim]
    df_ph1   = df_ts.loc[ph1_idx] if not ph1_idx.empty else pd.DataFrame(columns=df_ts.columns)
    df_ph2   = df_ts.loc[ph2_idx] if not ph2_idx.empty else pd.DataFrame(columns=df_ts.columns)
    hbonds   = find_hbonds(df_ts)
    n1, n2   = len(df_ph1), len(df_ph2)

    ph1_freqs, ph2_freqs = {}, {}
    for hb in hbonds:
        ph1_freqs[hb] = (df_ph1[hb] > 0).sum() / n1 * 100 if n1 > 0 else 0.0
        ph2_freqs[hb] = (df_ph2[hb] > 0).sum() / n2 * 100 if n2 > 0 else 0.0

    all_phase_frequencies[f'{sim}_ph1'] = ph1_freqs
    all_phase_frequencies[f'{sim}_ph2'] = ph2_freqs

freq_table   = pd.DataFrame(all_phase_frequencies)
existing     = [c for c in PHASES_LIST_ORDERED if c in freq_table.columns]
freq_table   = freq_table[existing]
freq_table   = freq_table[freq_table.sum(axis=1) > 0]
filtered_ft  = freq_table[(freq_table > 80).any(axis=1)]
print(f'Bonds with ≥ 80% frequency in any phase: {len(filtered_ft)}')

In [ ]:
# 5. Decision Tree on frequency table (graphviz rendering)
X = filtered_ft.T
y = X.index
feature_cols = X.columns

clf = DecisionTreeClassifier(max_depth=4, random_state=42)
clf.fit(X, y)

if _PYDOT_AVAILABLE:
    dot_data = StringIO()
    export_graphviz(clf, out_file=dot_data, filled=True, rounded=True,
                    special_characters=True, feature_names=feature_cols,
                    class_names=y.unique().tolist(), impurity=False, proportion=False)
    graph = pydotplus.graph_from_dot_data(dot_data.getvalue())
    graph.write_png(os.path.join(OUTPUT_DIR, 'decision_tree_V12_V7.png'))
    display(Image(graph.create_png()))
else:
    fig, ax = plt.subplots(figsize=(16, 6))
    plot_tree(clf, feature_names=feature_cols, class_names=list(y),
              filled=True, rounded=True, impurity=False, ax=ax)
    plt.tight_layout()
    plt.show()

In [ ]:
# 6. Dendrogram — simulation phases by H-bond frequency
linked = linkage(X, 'complete')
plt.figure(figsize=(10, 7))
dendrogram(linked, orientation='top', labels=y.tolist(), distance_sort='descending')
plt.title('Clustering of Simulation Phases by Interaction Frequencies')
plt.tight_layout()
plt.show()

In [ ]:
# 7. RMSD-based clustering — simulation level
os.chdir(ROOTDIR_TABLES)
rmsd_df = pd.read_csv('trans_st_RMSD.csv')
df_sele = rmsd_df[['V12_RMSD', 'V7_RMSD']]
X_rmsd  = df_sele.T.to_numpy()

linked_rmsd = linkage(X_rmsd, 'ward')
plt.figure(figsize=(10, 7))
dendrogram(linked_rmsd, orientation='top', labels=df_sele.columns.tolist(),
           distance_sort='descending')
plt.title('Clustering of Simulation Phases by RMSD')
plt.tight_layout()
plt.show()

In [ ]:
# 8. RMSD-based clustering — individual frames
frames_data, frames_labels = [], []
for i in range(len(rmsd_df)):
    row = rmsd_df.loc[i]
    for sim, rmsd_col, frames_col in [('V7', 'V7_RMSD', 'V7_frames'),
                                       ('V12', 'V12_RMSD', 'V12_frames')]:
        frame  = row[frames_col]
        rmsd   = row[rmsd_col]
        phase  = 'ph1' if frame <= PHASE_LIMIT_DICO[sim] else 'ph2'
        frames_data.append({'RMSD_Value': rmsd})
        frames_labels.append(f'{sim}_F{int(frame)}_{phase}')

X_frames = pd.DataFrame(frames_data)
linked_frames = linkage(X_frames['RMSD_Value'].to_numpy().reshape(-1, 1), 'ward')

fig = plt.figure(figsize=(10, 20))
dendrogram(linked_frames, orientation='left', labels=frames_labels,
           leaf_font_size=5, leaf_rotation=0)
plt.title('Individual Frame Clustering by RMSD')
plt.tight_layout()
plt.show()
fig.savefig(os.path.join(OUTPUT_DIR, 'RMSD_frames.png'), dpi=300, bbox_inches='tight')

In [ ]:
# 9. PCA on per-frame interaction data (phase CSVs must exist)
csv_to_read = ['V7_ph1_interactions.csv', 'V7_ph2_interactions.csv',
               'V12_ph1_interactions.csv', 'V12_ph2_interactions.csv']

os.chdir(ROOTDIR_TABLES)
df_full = pd.DataFrame()
for fname in csv_to_read:
    if not os.path.exists(fname):
        print(f'[WARN] {fname} not found — skipping.')
        continue
    parts   = fname.split('_')
    label   = f'{parts[0]}_{parts[1]}'
    df_temp = pd.read_csv(fname)
    df_temp = df_temp[find_hbonds(df_temp)]
    df_temp['label'] = label
    df_full = pd.concat([df_full, df_temp])

if not df_full.empty:
    # Remove all-zero rows and columns
    df_full = df_full.loc[(df_full.drop('label', axis=1) != 0).any(axis=1)]
    for col in df_full.columns.copy():
        if col != 'label' and df_full[col].sum() == 0:
            df_full.drop(columns=[col], inplace=True)

    X_pca = df_full.drop('label', axis=1)
    y_pca = df_full['label']
    X_sc  = StandardScaler().fit_transform(X_pca)
    pca   = PCA(n_components=2)
    comps = pca.fit_transform(X_sc)
    var   = pca.explained_variance_ratio_ * 100

    pca_result = pd.DataFrame(comps, columns=['PC1', 'PC2'])
    pca_result['label'] = y_pca.reset_index(drop=True)
    fig = px.scatter(pca_result, x='PC1', y='PC2', color='label',
                     title=f'PCA — PC1 ({var[0]:.1f}%) vs PC2 ({var[1]:.1f}%)')
    fig.update_traces(marker=dict(size=8, opacity=0.7))
    fig.show()
else:
    print('[INFO] No phase CSVs found — PCA skipped.')